# USA Economy Analysis

## 02-data-preparation: How do we clean and prepare it

Preparing raw economic data for analysis:

- Standardizing column names
- Converting date columns to datetime
- Organizing data by year and quarter
- Converting monthly data to quarterly frequency
- Merging economic datasets
- Creating a clean and analysis-ready dataset

## Imports

In [242]:
import pandas as pd
from pathlib import Path

## Loading Data

Load the raw economic datasets from the project's raw data directory.

In [243]:
DATA_PATH =Path("../data/raw")

In [244]:
df_gdp = pd.read_csv(DATA_PATH / "gdp_data.csv")

In [245]:
df_cpi = pd.read_csv(DATA_PATH / "cpi_data.csv")

In [246]:
df_unemployment = pd.read_csv(DATA_PATH / "unemployment_data.csv")

## Initial Inspection

Inspect the structure and data types of the raw datasets before applying transformations.

In [247]:
print(df_gdp.shape)
print(df_cpi.shape)
print(df_unemployment.shape)

(201, 2)
(601, 2)
(601, 2)


In [248]:
df_gdp.head()

,DATE,GDP
0,1974-01-01,1491.209
1,1974-04-01,1530.056
2,1974-07-01,1560.026
3,1974-10-01,1599.679
4,1975-01-01,1616.116


In [249]:
df_gdp.tail()

,DATE,GDP
196,2023-01-01,26813.601
197,2023-04-01,27063.012
198,2023-07-01,27610.128
199,2023-10-01,27956.998
200,2024-01-01,28269.174


In [250]:
df_cpi.head()

,DATE,CPIAUCSL
0,1974-01-01,46.8
1,1974-02-01,47.3
2,1974-03-01,47.8
3,1974-04-01,48.1
4,1974-05-01,48.6


In [251]:
df_cpi.tail()

,DATE,CPIAUCSL
596,2023-09-01,307.288
597,2023-10-01,307.531
598,2023-11-01,308.024
599,2023-12-01,308.742
600,2024-01-01,309.685


In [252]:
df_unemployment.head()

,DATE,UNRATE
0,1974-01-01,5.1
1,1974-02-01,5.2
2,1974-03-01,5.1
3,1974-04-01,5.1
4,1974-05-01,5.1


In [253]:
df_unemployment.tail()

,DATE,UNRATE
596,2023-09-01,3.8
597,2023-10-01,3.8
598,2023-11-01,3.7
599,2023-12-01,3.7
600,2024-01-01,3.7


In [254]:
df_gdp.dtypes

DATE        str
GDP     float64
dtype: object

In [255]:
df_cpi.dtypes

DATE            str
CPIAUCSL    float64
dtype: object

In [256]:
df_unemployment.dtypes

DATE          str
UNRATE    float64
dtype: object

## Rename Columns

Standardize column names using lowercase and descriptive names.

In [257]:
df_gdp = df_gdp.rename(columns={
    "GDP": "gdp",
    "DATE":"date"
})

In [258]:
df_cpi = df_cpi.rename(columns={
    "DATE":"date",
    "CPIAUCSL":"cpi"
})

In [259]:
df_unemployment = df_unemployment.rename(columns={
    "DATE":"date",
    "UNRATE":"unemployment_rate"
})

## Convert Date to Datetime

Convert the date columns from strings to pandas datetime objects.

In [260]:

df_gdp["date"] = pd.to_datetime(df_gdp["date"])

In [261]:
df_cpi["date"] = pd.to_datetime(df_cpi["date"])

In [262]:
df_unemployment["date"] = pd.to_datetime(df_unemployment["date"])

## Sort Data by Date

In [263]:
df_gdp = df_gdp.sort_values("date")
df_cpi = df_cpi.sort_values("date")
df_unemployment = df_unemployment.sort_values("date")

## Reset Index

Reset the index after sorting the datasets.

In [264]:
df_gdp = df_gdp.reset_index(drop= True)
df_cpi = df_cpi.reset_index(drop= True)
df_unemployment = df_unemployment.reset_index(drop= True)

## Convert Data to Quarterly Frequency

### GDP

GDP is already reported at a quarterly frequency. Therefore, no aggregation is required. Year and quarter are extracted from the date column to create a common time identifier.

In [265]:
df_gdp["year"] = df_gdp["date"].dt.year
df_gdp["quarter"] = df_gdp["date"].dt.quarter

In [266]:
df_gdp_quarterly = df_gdp[["year", "quarter", "gdp"]].copy()

In [267]:
df_gdp_quarterly["year-quarter"] = (
    df_gdp_quarterly["year"].astype(str)
    + "Q"
    + df_gdp_quarterly["quarter"].astype(str)
)
df_gdp_quarterly.insert(2,"year-quarter",df_gdp_quarterly.pop("year-quarter"))

In [268]:
df_gdp_quarterly

,year,quarter,year-quarter,gdp
0,1974,1,1974Q1,1491.209
1,1974,2,1974Q2,1530.056
2,1974,3,1974Q3,1560.026
3,1974,4,1974Q4,1599.679
4,1975,1,1975Q1,1616.116
...,...,...,...,...
196,2023,1,2023Q1,26813.601
197,2023,2,2023Q2,27063.012
198,2023,3,2023Q3,27610.128
199,2023,4,2023Q4,27956.998


### CPI

CPI is reported monthly. Monthly observations are aggregated to quarterly frequency using the arithmetic mean of the three monthly observations within each quarter.

In [269]:
df_cpi["quarter"] = df_cpi["date"].dt.quarter
df_cpi["year"] = df_cpi["date"].dt.year

In [270]:
df_cpi_quarterly = df_cpi.groupby(["year","quarter"])["cpi"].mean()
df_cpi_quarterly = pd.DataFrame(df_cpi_quarterly)
df_cpi_quarterly = df_cpi_quarterly.reset_index()

In [271]:
df_cpi_quarterly["year-quarter"] = (
    df_cpi_quarterly["year"].astype(str)
    + "Q"
    + df_cpi_quarterly["quarter"].astype(str)
)
df_cpi_quarterly.insert(2,"year-quarter",df_cpi_quarterly.pop("year-quarter"))

In [272]:
df_cpi_quarterly

,year,quarter,year-quarter,cpi
0,1974,1,1974Q1,47.300000
1,1974,2,1974Q2,48.566667
2,1974,3,1974Q3,49.933333
3,1974,4,1974Q4,51.466667
4,1975,1,1975Q1,52.566667
...,...,...,...,...
196,2023,1,2023Q1,301.203000
197,2023,2,2023Q2,303.466667
198,2023,3,2023Q3,306.034333
199,2023,4,2023Q4,308.099000


### Unemployment Rate

The unemployment rate is reported monthly. Monthly observations are aggregated to quarterly frequency using the arithmetic mean of the three monthly observations within each quarter.

In [273]:
df_unemployment["year"] = df_unemployment["date"].dt.year
df_unemployment["quarter"] = df_unemployment["date"].dt.quarter


In [274]:
df_unemployment_quarterly = df_unemployment.groupby(["year","quarter"])["unemployment_rate"].mean()
df_unemployment_quarterly = pd.DataFrame(df_unemployment_quarterly)
df_unemployment_quarterly = df_unemployment_quarterly.reset_index()

In [275]:
df_unemployment_quarterly["year-quarter"] = (
    df_unemployment_quarterly["year"].astype(str)
    + "Q"
    + df_unemployment_quarterly["quarter"].astype(str)
)
df_unemployment_quarterly.insert(2,"year-quarter",df_unemployment_quarterly.pop("year-quarter"))

In [276]:
df_unemployment_quarterly

,year,quarter,year-quarter,unemployment_rate
0,1974,1,1974Q1,5.133333
1,1974,2,1974Q2,5.200000
2,1974,3,1974Q3,5.633333
3,1974,4,1974Q4,6.600000
4,1975,1,1975Q1,8.266667
...,...,...,...,...
196,2023,1,2023Q1,3.500000
197,2023,2,2023Q2,3.566667
198,2023,3,2023Q3,3.700000
199,2023,4,2023Q4,3.733333


## Merge the Datasets

Merge GDP, CPI, and unemployment data using the common year and quarter identifiers.

An inner join is used to keep only quarters that are available across all three datasets.

In [277]:
df_economy = df_gdp_quarterly.merge(df_cpi_quarterly,
                                    on=["year","quarter","year-quarter"],
                                    how="inner")

In [278]:
df_economy = df_economy.merge(df_unemployment_quarterly,
on=["year","quarter","year-quarter"],
how= "inner"
)

In [279]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate
0,1974,1,1974Q1,1491.209,47.300000,5.133333
1,1974,2,1974Q2,1530.056,48.566667,5.200000
2,1974,3,1974Q3,1560.026,49.933333,5.633333
3,1974,4,1974Q4,1599.679,51.466667,6.600000
4,1975,1,1975Q1,1616.116,52.566667,8.266667
...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000
197,2023,2,2023Q2,27063.012,303.466667,3.566667
198,2023,3,2023Q3,27610.128,306.034333,3.700000
199,2023,4,2023Q4,27956.998,308.099000,3.733333


## Final Validation

Check the prepared dataset for missing values, duplicate observations, dimensions, and data types.

In [280]:
df_economy.isnull().sum()

year                 0
quarter              0
year-quarter         0
gdp                  0
cpi                  0
unemployment_rate    0
dtype: int64

In [281]:
df_economy.duplicated().sum()

np.int64(0)

In [282]:
df_economy.shape

(201, 6)

In [283]:
df_economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               201 non-null    int32  
 1   quarter            201 non-null    int32  
 2   year-quarter       201 non-null    str    
 3   gdp                201 non-null    float64
 4   cpi                201 non-null    float64
 5   unemployment_rate  201 non-null    float64
dtypes: float64(3), int32(2), str(1)
memory usage: 9.2 KB


## Final Inspection

Inspect the first and last observations of the final quarterly dataset.

In [284]:
df_economy.head()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate
0,1974,1,1974Q1,1491.209,47.300000,5.133333
1,1974,2,1974Q2,1530.056,48.566667,5.200000
2,1974,3,1974Q3,1560.026,49.933333,5.633333
3,1974,4,1974Q4,1599.679,51.466667,6.600000
4,1975,1,1975Q1,1616.116,52.566667,8.266667


In [285]:
df_economy.tail()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate
196,2023,1,2023Q1,26813.601,301.203000,3.500000
197,2023,2,2023Q2,27063.012,303.466667,3.566667
198,2023,3,2023Q3,27610.128,306.034333,3.700000
199,2023,4,2023Q4,27956.998,308.099000,3.733333
200,2024,1,2024Q1,28269.174,309.685000,3.700000


## Save Prepared Dataset

Save the prepared quarterly dataset in Feather format for efficient use in subsequent notebooks.

In [286]:
PROCESSED_PATH = Path("../data/processed")

df_economy.to_feather(
    PROCESSED_PATH / "economy_quarterly.feather"
)

## Preparation Summary

The raw GDP, CPI, and unemployment datasets were standardized and transformed into a common quarterly frequency. Monthly CPI and unemployment observations were aggregated using quarterly means, while GDP was already available at the quarterly frequency.

The resulting dataset contains 201 quarterly observations from 1974Q1 to 2024Q1 and is ready for feature engineering.